# Tutorial: Running the Optimized 4D Whole Cell Model on the Delta Gateway

**RESOURCES**

**Primary reference:** [Thornburg Z et. al, Bringing the genetically minimal cell to life on a computer in 4D, Cell 2026](https://www.cell.com/cell/fulltext/S0092-8674(26)00174-1)

**Code:** copy from `/projects/bgvl/containers/4DWCM_Gateway/Optimize_4DWCM_Minimal_Cell/` on Delta (section 1 — no GitHub login required).

**About:** This notebook launches the **optimized** 4DWCM (~3× faster than the paper baseline) inside the Gateway **4DCell Optimized** container.

**Website:** [4D Minimal Cell](https://minimalcell4d.web.illinois.edu/home/)

**Gateway session requirements:**
- **GPU Environment:** `4DCell Optimized`
- **GPUs:** 2 (RDME/CME on GPU 0, async DNA on GPU 1)
- **Jupyter kernel:** `LM 2.5 (Python 3.7)` — kernel name is legacy; the container ships LM 2.6 + btree_chromo 2.0

**Before starting:** run section 1 — it copies `Optimize_4DWCM_Minimal_Cell` from `/projects/bgvl/containers/4DWCM_Gateway/` (no GitHub login required on `bgvl`).

**Quick map (default test run):**
| What | Path |
|------|------|
| Your copy of the code | `/home/user/workspace/Optimize_4DWCM_Minimal_Cell/` |
| **Simulation outputs** (trajectories, counts, DNA) | `.../Data/<OUTPUT_DIR>/` |
| **Run log** (stdout/stderr text) | `.../logs/<logfile>.log` |
| Model inputs (read-only) | `.../input_data/` |
| Shared bundle (do not write here) | `/projects/bgvl/containers/4DWCM_Gateway/Optimize_4DWCM_Minimal_Cell` |

---
## 1. Copy simulation code and set paths

Run the **next two cells** (copy, then paths). The copy cell uses `rsync` from the shared `bgvl` bundle into your workspace (~13 MB; code + `input_data` only).

**Important:** copy into your workspace, not a symlink — the simulation creates a large writable `Data/` tree beside the code. Nothing is written back to `/projects/bgvl/containers/4DWCM_Gateway/`.

In [ ]:
%%bash
WORKSPACE=/home/user/workspace
SHARED=/projects/bgvl/containers/4DWCM_Gateway/Optimize_4DWCM_Minimal_Cell
DEST="$WORKSPACE/Optimize_4DWCM_Minimal_Cell"

mkdir -p "$WORKSPACE"

if [ -f "$DEST/Whole_Cell_Minimal_Cell.py" ]; then
  echo "Already present: $DEST/Whole_Cell_Minimal_Cell.py"
elif [ ! -f "$SHARED/Whole_Cell_Minimal_Cell.py" ]; then
  echo "ERROR: shared bundle not found at $SHARED"
  echo "Are you on the bgvl allocation with /projects/bgvl mounted?"
  exit 1
else
  mkdir -p "$DEST"
  rsync -rl --no-perms --no-owner --no-group "$SHARED/" "$DEST/"
  echo "Copied $SHARED -> $DEST (rsync; no permission preserve)"
fi

ls -la "$DEST/Whole_Cell_Minimal_Cell.py"

In [ ]:
import os
import subprocess
from pathlib import Path

ENTRY = "Whole_Cell_Minimal_Cell.py"
REPO_NAME = "Optimize_4DWCM_Minimal_Cell"
WORKSPACE = Path("/home/user/workspace")
SHARED = Path("/projects/bgvl/containers/4DWCM_Gateway/Optimize_4DWCM_Minimal_Cell")

def find_repo():
    roots = []
    for p in (WORKSPACE, Path("/home/user"), Path(os.environ.get("HOME", "/home/user"))):
        p = p.expanduser()
        if p not in roots:
            roots.append(p)
    for root in roots:
        repo = root / REPO_NAME
        if (repo / ENTRY).is_file():
            return repo
    return WORKSPACE / REPO_NAME

def ensure_repo():
    repo = find_repo()
    if (repo / ENTRY).is_file():
        return repo
    if not (SHARED / ENTRY).is_file():
        print(f"ERROR: shared bundle not found at {SHARED}")
        print("Is /projects/bgvl mounted in this Gateway session?")
        return repo
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    dest = WORKSPACE / REPO_NAME
    dest.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "rsync", "-rl", "--no-perms", "--no-owner", "--no-group",
        str(SHARED) + "/", str(dest) + "/",
    ], check=True)
    print(f"Copied {SHARED} -> {dest} (rsync; no permission preserve)")
    return dest

REPO = ensure_repo()
HOME = REPO.parent
LOGDIR = REPO / "logs"
REPO_PIN = Path("/tmp/4dwcm_gateway_repo.txt")

if (REPO / ENTRY).is_file():
    LOGDIR.mkdir(parents=True, exist_ok=True)
    REPO_PIN.write_text(str(REPO))
else:
    print("WARN: Whole_Cell_Minimal_Cell.py still missing after copy attempt")

# Optimized stack paths (match 4DCell Optimized container)
DNA_SOFTWARE = "/Software/opt"
PYLM_PATH = "/Software/Lattice_Microbes_2.6/src/pylm"

os.environ["TMPDIR"] = "/tmp"
os.environ["HOME"] = str(HOME)
os.environ["PATH"] = "/opt/conda/envs/lm_2.5_dev/bin:" + os.environ.get("PATH", "")
os.environ["PYTHONPATH"] = PYLM_PATH + os.pathsep + os.environ.get("PYTHONPATH", "")
os.environ["LD_LIBRARY_PATH"] = "/usr/local/lib64:/usr/local/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["DNA_GPU_ID"] = "1"          # async DNA on second GPU
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# Short test defaults (edit for a production run)
OUTPUT_DIR = "4dwcm_10s"
SIM_TIME = 10          # biological seconds; use 7200 for full cell cycle
RNG_SEED = 13
RDME_GPU = 0

DATA_DIR = REPO / "Data" / OUTPUT_DIR
INPUT_DIR = REPO / "input_data"
RUN_LOG = LOGDIR / "run_4dwcm_10s.log"
RUN_PARAMS = Path("/tmp/4dwcm_gateway_run.env")
RUN_PARAMS.write_text(
    f"OUTPUT_DIR={OUTPUT_DIR}\nSIM_TIME={SIM_TIME}\nRNG_SEED={RNG_SEED}\nRDME_GPU={RDME_GPU}\n"
    f"RUN_LOG={RUN_LOG}\n"
)

print("=" * 60)
print("Paths (edit OUTPUT_DIR / SIM_TIME above for new runs)")
print("=" * 60)
print("REPO:          ", REPO)
print("Entry script:  ", REPO / ENTRY, "->", (REPO / ENTRY).is_file())
print("INPUT_DIR:     ", INPUT_DIR, "(kinetic params, genome — read-only)")
print("DATA_DIR:      ", DATA_DIR, "(simulation OUTPUT — created during run)")
print("RUN_LOG:       ", RUN_LOG, "(text log — not science data)")
print("DNA_SOFTWARE:  ", DNA_SOFTWARE)
print("PYTHONPATH:    ", os.environ["PYTHONPATH"])

---
### Where outputs vs. logs go

The simulation produces **two different kinds of files**:

#### 1. Simulation data (`Data/<OUTPUT_DIR>/`) — the science output

Created by `Whole_Cell_Minimal_Cell.py` as it runs. Default test folder:

```
/home/user/workspace/Optimize_4DWCM_Minimal_Cell/Data/4dwcm_10s/
```

| Subfolder / file | What it is |
|------------------|------------|
| `counts_and_fluxes.csv` | Merged metabolism counts and fluxes over time |
| `counts_fluxes_temp/` | Per-hook CSV chunks (merged into the file above) |
| `DNA/` | Chromosome structures, LAMMPS data, btree_chromo logs |
| `CME/` | CME-side outputs |
| `fluxes/` | Flux trajectory files |
| `restart_files/` | Checkpoints (`plattice.npy`, region arrays) for `-mh` stop / restart |
| `*.lm` (in run folder) | RDME lattice trajectory files |

Change `OUTPUT_DIR` in the paths cell (e.g. `my_run_7200`) to avoid overwriting a previous run. A full 7200 s run can be **tens of GB**.

#### 2. Run log (`logs/*.log`) — text for humans

```
/home/user/workspace/Optimize_4DWCM_Minimal_Cell/logs/run_4dwcm_10s.log
```

Because the simulation runs in the **background** (`nohup ... &`), all printed messages (CME timings, ODE compile, errors, `Simulation completed!`) go to this **text file**, not into the notebook cell. Use the tail cells in section 4 to read it.

The log is for **monitoring and debugging**. Analysis uses files under `Data/`, not the log.

> **Persistence:** outputs live in your Gateway workspace. Copy important `Data/` folders to `/projects/bgvl/<your_username>/` before ending a session if you need long-term storage.

---
### Environment check (optional)

Runs **after** section 1. Package imports use `/opt/conda/envs/lm_2.5_dev/bin/python` (same as the simulation), not the Jupyter kernel — missing `jLM`/`pyLM` in the kernel alone is normal.

If **Hook.py** is missing, re-run the copy cell in section 1.


In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

SIM_PYTHON = "/opt/conda/envs/lm_2.5_dev/bin/python"
check_env = os.environ.copy()
check_env["PYTHONPATH"] = PYLM_PATH

print("=" * 60)
print("Environment check (4DCell Optimized)")
print("=" * 60)
print(f"\nJupyter kernel: {sys.executable} ({sys.version.split()[0]})")
print("  (packages below are checked with the simulation Python, not this kernel)")
print(f"Simulation Python: {SIM_PYTHON}")

print("\n--- Python packages (simulation env) ---")
if not os.path.isfile(SIM_PYTHON):
    print(f"  WARN: {SIM_PYTHON} not found — are you in 4DCell Optimized?")
else:
    for mod in ("jLM", "pyLM", "lm", "odecell"):
        cmd = [
            SIM_PYTHON, "-c",
            f"import {mod}; print(getattr({mod}, '__file__', 'ok'))",
        ]
        r = subprocess.run(cmd, env=check_env, capture_output=True, text=True)
        if r.returncode == 0:
            print(f"  {mod:8s} OK  ({r.stdout.strip()})")
        else:
            err = (r.stderr or r.stdout or "import failed").strip().splitlines()[-1]
            print(f"  {mod:8s} MISSING — {err}")

print("\n--- Optimized fork markers ---")
hook = REPO / "Hook.py"
if hook.is_file():
    txt = hook.read_text()
    print("  setSolverCached:", "setSolverCached" in txt)
    print("  Cython ODE on:  ", "_ode_use_cython = True" in txt)
else:
    print(f"  WARN: {hook} not found")
    print("  Copy first: rsync from /projects/bgvl/containers/4DWCM_Gateway/Optimize_4DWCM_Minimal_Cell (see section 1)")

print("\n--- DNA / LAMMPS stack ---")
for label, paths in (
    ("btree_chromo", [f"{DNA_SOFTWARE}/btree_chromo/build/apps/btree_chromo"]),
    ("gen_sc_chain", [f"{DNA_SOFTWARE}/sc_chain_generation/fortran/gen_sc_chain",
                       "/Software/sc_chain_generation/fortran/gen_sc_chain"]),
    ("LAMMPS lib", ["/usr/local/lib/liblammps_delta_kokkos.so.0",
                     "/usr/local/lib64/liblammps_delta_kokkos.so.0"]),
):
    p = next((x for x in paths if os.path.isfile(x)), None)
    print(f"  {label:14s} {p or 'NOT FOUND'}")

print("\n--- GPUs ---")
nvsmi = shutil.which("nvidia-smi")
if nvsmi:
    subprocess.run([nvsmi, "-L"], check=False)
else:
    print("  nvidia-smi not on PATH (common in Jupyter); checking fallbacks")
    cvd = os.environ.get("CUDA_VISIBLE_DEVICES", "(unset)")
    print(f"  CUDA_VISIBLE_DEVICES: {cvd}")
    dev_nodes = sorted(Path("/dev").glob("nvidia*"))
    print(f"  /dev/nvidia* nodes: {', '.join(p.name for p in dev_nodes) or '(none)'}")
    drv = Path("/proc/driver/nvidia/version")
    if drv.is_file():
        print(f"  driver: {drv.read_text().splitlines()[0]}")
print("\n" + "=" * 60)

---
## 2. Name the run log

The simulation runs in the **background**, so notebook cells will not show live output. Instead, everything printed to the terminal is captured in a **log file** under `logs/`.

| File | Role |
|------|------|
| `logs/run_4dwcm_10s.log` | Human-readable trace: hook timings, CME/ODE/DNA messages, errors, `Simulation completed!` |
| `Data/4dwcm_10s/` | Actual simulation products (CSV, DNA, lattice trajectories, checkpoints) |

Edit `RUN_LOG` below if you start a new run — use a **different log name** each time so you do not overwrite a previous log. `RUN_LOG` is already set in the paths cell; this cell exposes it for the tail cells below.


In [ ]:
# Set in section 1 paths cell; change name for each new run
RUN_LOG = LOGDIR / "run_4dwcm_10s.log"
print("Log file (monitoring only):", RUN_LOG)
print("Science output directory:  ", DATA_DIR)

---
## 3. Start the simulation

> **Requires 2 GPUs** allocated in the Gateway form (**`4DCell Optimized`**). RDME/CME use GPU 0 (`-cd 0`); DNA runs async on GPU 1 (`DNA_GPU_ID=1`). A 1-GPU session will not run the optimized stack correctly.

---
### Command-line arguments (`Whole_Cell_Minimal_Cell.py`)

| Variable | Shorthand | Description |
|----------|-----------|-------------|
| `--outputDir` | `-od` | Name of directory (under `Data/`) to store trajectories. |
| `--simTime` | `-t` | Biological time to simulate, in **seconds** (7200 = full cell cycle) |
| `--cudaDevices` | `-cd` | GPU index for RDME + CME (use **0**) |
| `--dnaSoftwareDirectory` | `-dsd` | DNA software root — use **`/Software/opt/`** in 4DCell Optimized |
| `--dnaRngSeed` | `-drs` | Integer RNG seed for the chromosome / DNA pipeline |
| `--maximumHours` | `-mh` | Optional wall-clock limit (hours); checkpoints then stops cleanly |
| `--workingDirectory` | `-wd` | Base directory of the run (default: current working directory) |

**Environment (set in section 1):**
- `PYTHONPATH=/Software/Lattice_Microbes_2.6/src/pylm`
- `DNA_GPU_ID=1` — btree_chromo runs on the **second** GPU (async overlap)
- `LD_LIBRARY_PATH` must include `/usr/local/lib` (Kokkos LAMMPS)

**Example (gateway, background + log):**

```bash
export PYTHONPATH=/Software/Lattice_Microbes_2.6/src/pylm
export DNA_GPU_ID=1
export LD_LIBRARY_PATH=/usr/local/lib64:/usr/local/lib
nohup python -u Whole_Cell_Minimal_Cell.py \
  -od 4dwcm_10s -t 10 -cd 0 -drs 13 -dsd /Software/opt/ \
  > logs/run.log 2>&1 &
```

Edit **`OUTPUT_DIR`**, **`SIM_TIME`**, and **`RUN_LOG`** in section 1 / 2. Defaults: `OUTPUT_DIR=4dwcm_10s`, `SIM_TIME=10` (short test). A full 7200 s cell cycle takes **~25 h wall** on 2× A100.

**Where results land:**
- Science data → `Data/<OUTPUT_DIR>/` (see table in section 1)
- Text log → `logs/<your_log>.log` (section 4 tails this file)

> **Gateway session limit:** request **48 hours** in the allocation form for a full 7200 s run (~25 h wall). For short tests, 4 hours is enough. Pass `-mh` slightly below your wall limit so `restart_files/` are written if the job ends early.

In [ ]:
%%bash
set -e
if [ -f /tmp/4dwcm_gateway_repo.txt ]; then
  REPO="$(cat /tmp/4dwcm_gateway_repo.txt)"
else
  REPO=""
  for d in /home/user/workspace/Optimize_4DWCM_Minimal_Cell /home/user/Optimize_4DWCM_Minimal_Cell; do
    if [ -f "$d/Whole_Cell_Minimal_Cell.py" ]; then REPO="$d"; break; fi
  done
fi
ENTRY="$REPO/Whole_Cell_Minimal_Cell.py"

if [ -z "$REPO" ] || [ ! -f "$ENTRY" ]; then
  echo "ERROR: Whole_Cell_Minimal_Cell.py not found. Re-run section 1 copy cell."
  exit 1
fi

# Run parameters from section 1 paths cell
if [ -f /tmp/4dwcm_gateway_run.env ]; then
  source /tmp/4dwcm_gateway_run.env
else
  OUTPUT_DIR=4dwcm_10s
  SIM_TIME=10
  RNG_SEED=13
  RDME_GPU=0
  RUN_LOG="$REPO/logs/run_4dwcm_10s.log"
fi

echo "Using REPO:       $REPO"
echo "OUTPUT_DIR:     $OUTPUT_DIR  ->  Data/$OUTPUT_DIR/"
echo "SIM_TIME:       $SIM_TIME s (biological)"
echo "RUN_LOG:        $RUN_LOG"

mkdir -p "$REPO/logs"
cd "$REPO"

export PYTHONPATH=/Software/Lattice_Microbes_2.6/src/pylm
export LD_LIBRARY_PATH=/usr/local/lib64:/usr/local/lib:${LD_LIBRARY_PATH:-}
export DNA_GPU_ID=1
export TMPDIR=/tmp
export HOME="$(dirname "$REPO")"
export HDF5_USE_FILE_LOCKING=FALSE
export XDG_CACHE_HOME=/tmp/.cache
export PYTHONPYCACHEPREFIX=/tmp/.pycache

nohup /opt/conda/envs/lm_2.5_dev/bin/python -u "$ENTRY" \
  -od "$OUTPUT_DIR" -t "$SIM_TIME" -cd "$RDME_GPU" -drs "$RNG_SEED" -dsd /Software/opt/ \
  > "$RUN_LOG" 2>&1 &
echo "Simulation started in background, PID: $!"
echo "Monitor log:  tail -f $RUN_LOG"
echo "Output data:  $REPO/Data/$OUTPUT_DIR/"

---
## 4. Track the progress

**Two places to look:**

1. **Log file** (`RUN_LOG`) — is the run healthy? Tail it below. Look for `Simulation completed!` or tracebacks.
2. **Data folder** (`DATA_DIR`) — science outputs appear here as the run progresses (section 4b below).

A **7200 s** optimized run takes **~25 h wall** on 2× A100. The first metabolism hook compiles the Cython ODE once (~60–90 s); later hooks are ~1 s each.

**Healthy log markers:**
- `CME_WORKER: persistent worker is ready`
- `Compiling cythonCompiledFunctions.pyx` (once)
- `ODE time: ~1` (after the first hook)
- `DNA time: ~0.5` s, no repeated `RESCUE`
- `Simulation completed!` at the end

### 4a. Tail the log (monitoring text)

In [ ]:
if RUN_LOG.is_file():
    print(f"--- last 80 lines of {RUN_LOG} ---\n")
    subprocess.run(["tail", "-n", "80", str(RUN_LOG)], check=False)
else:
    print(f"No log yet: {RUN_LOG}")
    print("Start the simulation in section 3, wait a few seconds, then re-run this cell.")

### 4b. List simulation output files (science data)

After the run finishes (or while it is running), check `DATA_DIR`. This is separate from the log — these files are what you analyze or archive.

In [ ]:
from pathlib import Path

print("Science output directory:", DATA_DIR)
if not DATA_DIR.is_dir():
    print("(not created yet — run still starting or OUTPUT_DIR unchanged)")
else:
    for p in sorted(DATA_DIR.iterdir()):
        if p.is_dir():
            n = sum(1 for _ in p.rglob("*") if _.is_file())
            print(f"  {p.name}/  ({n} files)")
        else:
            size_mb = p.stat().st_size / 1e6
            print(f"  {p.name}  ({size_mb:.2f} MB)")
    key = DATA_DIR / "counts_and_fluxes.csv"
    if key.is_file():
        print(f"\nKey output ready: {key}")

---
## 5. Restart the simulation (optional)

Run **only if the previous simulation failed** or you want to continue toward 7200 s after a checkpointed stop (`-mh`). Resumes from `Data/{output_dir}/` using **`Restart_Whole_Cell_Minimal_Cell.py`**.

Use the **same** `-od`, `-dsd /Software/opt/`, `PYTHONPATH`, and `DNA_GPU_ID=1` as the original run. `-t` is **additional** biological seconds (e.g. if you reached 3600 s and want 7200 s total, pass `-t 3600`).

**Quick check it actually failed:** run the tail cell above; if the last line shows a traceback and `pgrep -f Whole_Cell_Minimal_Cell.py` returns nothing, the run is dead.

In [ ]:
import subprocess

RESTART_LOG = LOGDIR / f"restart_{OUTPUT_DIR}.log"


In [ ]:
%%bash
set -e
if [ -f /tmp/4dwcm_gateway_repo.txt ]; then
  REPO="$(cat /tmp/4dwcm_gateway_repo.txt)"
else
  REPO=""
  for d in /home/user/workspace/Optimize_4DWCM_Minimal_Cell /home/user/Optimize_4DWCM_Minimal_Cell; do
    if [ -f "$d/Whole_Cell_Minimal_Cell.py" ]; then REPO="$d"; break; fi
  done
fi
ENTRY="$REPO/Restart_Whole_Cell_Minimal_Cell.py"

if [ -z "$REPO" ] || [ ! -f "$ENTRY" ]; then
  echo "ERROR: Restart_Whole_Cell_Minimal_Cell.py not found under $REPO"
  exit 1
fi

cd "$REPO"

if pgrep -f "Whole_Cell_Minimal_Cell.py" > /dev/null; then
  echo "ERROR: a Whole_Cell_Minimal_Cell.py process is still running."
  pgrep -af "Whole_Cell_Minimal_Cell.py" || true
  exit 1
fi

export PYTHONPATH=/Software/Lattice_Microbes_2.6/src/pylm
export LD_LIBRARY_PATH=/usr/local/lib64:/usr/local/lib:${LD_LIBRARY_PATH:-}
export DNA_GPU_ID=1
export TMPDIR=/tmp
export HOME="$(dirname "$REPO")"
export HDF5_USE_FILE_LOCKING=FALSE

if [ -f /tmp/4dwcm_gateway_run.env ]; then
  source /tmp/4dwcm_gateway_run.env
else
  OUTPUT_DIR=4dwcm_10s
  SIM_TIME=10
  RNG_SEED=13
  RDME_GPU=0
fi
ADDITIONAL_TIME="${ADDITIONAL_TIME:-$SIM_TIME}"
RESTART_LOG="$REPO/logs/restart_${OUTPUT_DIR}.log"

echo "Restart OUTPUT_DIR: $OUTPUT_DIR"
echo "Additional time:    ${ADDITIONAL_TIME}s"
echo "Restart log:        $RESTART_LOG"

nohup /opt/conda/envs/lm_2.5_dev/bin/python -u "$ENTRY" \
  -od "$OUTPUT_DIR" -t "$ADDITIONAL_TIME" -cd "$RDME_GPU" -drs "$RNG_SEED" -dsd /Software/opt/ \
  > "$RESTART_LOG" 2>&1 &
echo "Restart started in background, PID: $!"

In [ ]:

if RESTART_LOG.is_file():
    subprocess.run(["tail", "-n", "80", str(RESTART_LOG)], check=False)
else:
    print(f"No log yet: {RESTART_LOG}\n")

---

## 6. Cancel the simulation jobs

Run the next cell to list running simulation PIDs, then set **`JOB_PID`** in the following cell to send SIGTERM.

In [ ]:
%%bash
echo "Running jobs"
pgrep -af "Whole_Cell_Minimal_Cell.py" || echo "(none)"


In [ ]:
import subprocess

JOB_PID = 0  # set to integer PID from pgrep cell above, then re-run

if not JOB_PID:
    print("Set JOB_PID to an integer (the process id), then re-run this cell.")
else:
    r = subprocess.run(["kill", str(JOB_PID)], capture_output=True, text=True)
    if r.returncode == 0:
        print(f"Sent SIGTERM to PID {JOB_PID}.")
    else:
        print(f"kill failed (code {r.returncode}): {r.stderr or r.stdout or 'no such process or permission denied'}")